# 🦵 RSNA Knee Abnormality Detection — Deep EDA + Metric + CV Starter

**The one number that decides this competition: only ~1.3% of training studies are labeled.**
The rest come with a free-text *radiology report* (in many languages) that you must mine for labels.
So this is really a **weak-supervision / report-mining** problem wearing an imaging-competition coat.

This notebook covers, end to end:
1. The exact **metric** (macro-averaged ROC-AUC) — reproduced in code
2. The **labeled-vs-report-only** split (the crux)
3. **Label prevalence** & co-occurrence
4. **Report** languages & lengths
5. **Series / plane / sequence** structure (Sagittal·Coronal·Axial, fluid-sensitive, fat-sat)
6. **Slices per series** distribution
7. A **DICOM preview** across planes
8. A **study-grouped, label-stratified CV** you can copy

*If this helps your start, an upvote is very welcome 🙏 — questions in the comments.*

In [ ]:
import os, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi']=120; plt.rcParams['axes.grid']=True
pd.set_option('display.width',120)

COMP='/kaggle/input/competitions/rsna-knee-abnormality-detection'
if not os.path.isdir(COMP):
    # fallback: locate the mounted competition folder
    for r,d,_ in os.walk('/kaggle/input'):
        if 'train_series' in d: COMP=r; break
print('competition root:', COMP)
print(sorted(os.listdir(COMP))[:10])

LABELS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
        'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
train=pd.read_csv(f'{COMP}/train.csv')
series=pd.read_csv(f'{COMP}/train_series.csv')
print('train.csv', train.shape, '| train_series.csv', series.shape)

## 1. The metric — macro-averaged ROC-AUC (per study)
$$\text{Score}=\frac{1}{12}\sum_{i=1}^{12}\text{AUC}_i$$
Two consequences that should shape everything you do:
- It is a **ranking** metric per column — only the *order* of your scores matters, so calibration
  is irrelevant and **rank-averaging** is the natural way to ensemble.
- Every label is weighted **equally (1/12)** regardless of how rare it is — so a rare finding is
  worth just as much as a common one. Spend effort where the AUC *headroom* is largest.

In [ ]:
from sklearn.metrics import roc_auc_score
def macro_auc(y_true, y_pred):
    y_true=np.asarray(y_true,float); y_pred=np.asarray(y_pred,float)
    aucs=[]
    for j in range(y_true.shape[1]):
        col=y_true[:,j]
        if len(np.unique(col[~np.isnan(col)]))>1:
            m=~np.isnan(col); aucs.append(roc_auc_score(col[m],y_pred[m,j]))
    return float(np.mean(aucs))
# sanity: perfect=1, reversed=0, random~0.5
rng=np.random.default_rng(0); y=rng.integers(0,2,(500,12)); y[0]=0; y[1]=1
print('perfect :', round(macro_auc(y,y.astype(float)),3))
print('reversed:', round(macro_auc(y,1-y.astype(float)),3))
print('random  :', round(macro_auc(y,rng.random((500,12))),3))

## 2. The crux — how many studies are actually labeled?
Every study has a **Report**. Only a tiny fraction carry the 12 ground-truth labels.

In [ ]:
has_label = train[LABELS].notna().all(axis=1)
n_lab=int(has_label.sum()); n_tot=len(train)
print(f'Labeled studies : {n_lab} / {n_tot}  ({100*n_lab/n_tot:.1f}%)')
print(f'Report-only     : {n_tot-n_lab} / {n_tot}  ({100*(n_tot-n_lab)/n_tot:.1f}%)')
print('Studies with a Report:', int(train['Report'].notna().sum()))

fig,ax=plt.subplots(figsize=(6,1.6))
ax.barh([0],[n_lab],color='#2a9d8f',label=f'labeled ({n_lab})')
ax.barh([0],[n_tot-n_lab],left=[n_lab],color='#e9c46a',label=f'report-only ({n_tot-n_lab})')
ax.set_yticks([]); ax.set_xlabel('studies'); ax.legend(loc='center'); ax.grid(False)
ax.set_title('Labeled vs report-only — the whole game is turning reports into labels')
plt.show()

## 3. Label prevalence & co-occurrence (on the labeled subset)

In [ ]:
lab=train[has_label]
prev=lab[LABELS].mean().sort_values()
fig,ax=plt.subplots(figsize=(7,4))
ax.barh(prev.index, prev.values, color='#264653')
ax.set_xlabel('positive rate'); ax.set_title(f'Label prevalence (n={len(lab)} labeled studies)')
for i,v in enumerate(prev.values): ax.text(v+0.005,i,f'{v:.2f}',va='center',fontsize=8)
plt.tight_layout(); plt.show()

npos=lab[LABELS].sum(axis=1)
print('Findings per labeled study: mean %.1f, median %.0f, max %d'%(npos.mean(),npos.median(),npos.max()))
print('=> multi-label: most knees have several findings at once.')

## 4. Reports — multilingual & variable length

In [ ]:
rl=train['Report'].astype(str).str.len()
fig,ax=plt.subplots(figsize=(7,3))
ax.hist(rl.clip(upper=3500),bins=50,color='#e76f51')
ax.set_xlabel('report length (chars)'); ax.set_ylabel('# studies')
ax.set_title('Report length distribution'); plt.tight_layout(); plt.show()
print('length: median %d, p90 %d, max %d chars'%(rl.median(),rl.quantile(.9),rl.max()))
# crude language hint: share of non-ASCII reports
nonascii=train['Report'].astype(str).apply(lambda s: not s.isascii()).mean()
print(f'~{100*nonascii:.0f}% of reports contain non-ASCII characters (many languages present).')
print('\nExample (first 300 chars):\n', str(train.loc[has_label,"Report"].iloc[1])[:300])

## 5. Series structure — planes & sequence types
Each study is several MRI **series** (acquisitions). The per-series metadata tells you the plane
and whether the sequence is fluid-sensitive / fat-suppressed — useful to **route** pathologies
(e.g. ACL & meniscus read best on sagittal; effusion on fluid-sensitive sequences).

In [ ]:
sps=series.groupby('StudyInstanceUID').size()
print('series per study: mean %.1f, median %.0f, range %d-%d'%(sps.mean(),sps.median(),sps.min(),sps.max()))
fig,axes=plt.subplots(1,3,figsize=(12,3))
series['Anatomical_Plane'].value_counts().plot.bar(ax=axes[0],color='#2a9d8f',title='Anatomical_Plane')
series['Fluid_Sensitive'].value_counts().sort_index().plot.bar(ax=axes[1],color='#e9c46a',title='Fluid_Sensitive')
series['Fat_Suppression'].value_counts().sort_index().plot.bar(ax=axes[2],color='#e76f51',title='Fat_Suppression')
for a in axes: a.tick_params(axis='x',rotation=0)
plt.tight_layout(); plt.show()
print('\nPlane x Fluid_Sensitive:'); print(pd.crosstab(series.Anatomical_Plane, series.Fluid_Sensitive))

## 6. Slices per series

In [ ]:
# count .dcm per series folder for a sample of studies (fast)
sample_studies=train['StudyInstanceUID'].head(150)
cnts=[]
for st in sample_studies:
    for se in glob.glob(f'{COMP}/train_series/{st}/*'):
        n=len(glob.glob(se+'/*.dcm'))
        if n: cnts.append(n)
cnts=np.array(cnts)
fig,ax=plt.subplots(figsize=(7,3))
ax.hist(np.minimum(cnts,120),bins=40,color='#264653')
ax.set_xlabel('slices per series'); ax.set_ylabel('# series')
ax.set_title(f'Slices per series (sample of {len(cnts)} series): median {int(np.median(cnts))}')
plt.tight_layout(); plt.show()

## 7. DICOM preview — one study across planes

In [ ]:
import subprocess, sys
try:
    import gdcm  # noqa
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','python-gdcm','pylibjpeg','pylibjpeg-libjpeg','pylibjpeg-openjpeg'],check=False)
import pydicom
def load(p):
    d=pydicom.dcmread(p); a=d.pixel_array.astype(float)
    lo,hi=np.percentile(a,1),np.percentile(a,99); a=np.clip((a-lo)/(hi-lo+1e-6),0,1)
    if getattr(d,'PhotometricInterpretation','')=='MONOCHROME1': a=1-a
    return a
st=train['StudyInstanceUID'].iloc[0]
sers=series[series.StudyInstanceUID==st]
fig,axes=plt.subplots(1,min(4,len(sers)),figsize=(13,3.5))
axes=np.atleast_1d(axes)
for ax,(_,row) in zip(axes,sers.iterrows()):
    fs=sorted(glob.glob(f'{COMP}/train_series/{st}/{row.SeriesInstanceUID}/*.dcm'))
    if not fs: continue
    try: ax.imshow(load(fs[len(fs)//2]),cmap='gray')
    except Exception as e: ax.set_title('decode err'); continue
    ax.set_title(f'{row.Anatomical_Plane}\nFS={row.Fluid_Sensitive}'); ax.axis('off')
plt.suptitle('Mid-slice per series (one study)'); plt.tight_layout(); plt.show()

## 8. A study-grouped, label-stratified 5-fold CV
The unit scored is the **study** (there is no patient id linking studies), so plain multi-label
stratified K-fold over studies both matches the test structure and avoids leakage. Guarantee both
classes appear per label per fold, or a column's AUC is undefined.

In [ ]:
def mlskf(Y,k=5,seed=42):
    rng=np.random.default_rng(seed); n,L=Y.shape; folds=-np.ones(n,int)
    need=(Y.sum(0)[None,:]/k).repeat(k,0); cap=np.full(k,n/k)
    for lab in np.argsort(Y.sum(0)):
        pos=[i for i in np.where(Y[:,lab]==1)[0] if folds[i]<0]; rng.shuffle(pos)
        for i in pos:
            b=np.where(need[:,lab]==need[:,lab].max())[0]
            f=int(b[np.argmax(cap[b])]); folds[i]=f; need[f,lab]-=1; cap[f]-=1
    for i in np.where(folds<0)[0]:
        f=int(np.argmax(cap)); folds[i]=f; cap[f]-=1
    return folds
# demonstrate on the labeled subset
Y=lab[LABELS].astype(int).values; folds=mlskf(Y,5)
chk=pd.DataFrame({l:[int(Y[folds==f,j].sum()) for f in range(5)] for j,l in enumerate(LABELS)},
                 index=[f'fold{f}' for f in range(5)]).T
print('positives per label per fold (both-class check):'); print(chk)

## Takeaways
- **Labels, not model, decide this competition** — 98.7% of studies are report-only, so the biggest
  lever is how well you turn multilingual reports into labels.
- The metric is **rank-only macro-AUC** — ensemble by rank-averaging; don't sweat calibration.
- Rare labels count as much as common ones — chase **AUC headroom**.
- Route pathologies to the right **plane / sequence**; aggregate slice→study thoughtfully.

Good luck! ⭐ **Upvote if this saved you time** — happy to answer questions below.